# 04 — Multithreading with KOKKOS

lammps.js ships a second WebAssembly build with the LAMMPS
[KOKKOS](https://docs.lammps.org/Speed_kokkos.html) package, multithreaded
with pthreads. You enable it exactly like native LAMMPS — with command-line
arguments:

```python
lmp = await lammps(cmdargs=["-k", "on", "t", "4", "-sf", "kk"])
```

Threads need `SharedArrayBuffer`, which browsers only allow on
**cross-origin isolated** pages (COOP/COEP headers). The cell below checks
and falls back to the single-threaded build when isolation is off:

In [ ]:
%pip install lammps-js

In [ ]:
import js
from lammps import lammps

isolated = bool(getattr(js, "crossOriginIsolated", False))
print("cross-origin isolated:", isolated)

if isolated:
    lmp = await lammps(cmdargs=["-k", "on", "t", "4", "-sf", "kk"], output=None)
    print("KOKKOS package available:", lmp.has_package("KOKKOS"))
else:
    lmp = await lammps(output=None)
    print("running the single-threaded build instead")

## Benchmark

A larger LJ system, timed. On the KOKKOS build the pair, neighbor and
integration kernels run on 4 threads (`-sf kk` applies the `/kk` style
suffix automatically, same as native LAMMPS):

In [ ]:
import time

lmp.commands_string("""
units         lj
atom_style    atomic
lattice       fcc 0.8442
region        box block 0 12 0 12 0 12
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 1.44 87287
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
""")
print(lmp.get_natoms(), "atoms")

t0 = time.time()
lmp.command("run 100")
elapsed = time.time() - t0
print(f"100 steps: {elapsed:.2f} s "
      f"({lmp.get_natoms() * 100 / elapsed / 1e6:.2f} M atom-steps/s)")

lmp.close()

Re-run this notebook with isolation on and off (or vary `t 4`) to compare.
If this deployment is not cross-origin isolated, the same notebooks work
multithreaded when the site is served with

```
Cross-Origin-Opener-Policy: same-origin
Cross-Origin-Embedder-Policy: require-corp
```